In [4]:
# wfm_app.py
import streamlit as st
import pandas as pd
import numpy as np
import math
from ortools.sat.python import cp_model
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from io import BytesIO

st.set_page_config(layout="wide", page_title="Mini-WFM: Forecast→Staffing→Rostering")

# ----------------------
# Helpers: file parsing
# ----------------------
def load_master_turns(df):
    """
    Espera columnas: alguna de ['turno_id','id','nombre'] para id,
    'start'/'hora_ingreso'/'hora_inicio', 'end'/'hora_fin', 
    'break_minutes'/'break','almuerzo_minutes'/'lunch' (opcional).
    Los horarios pueden ser '08:00' o '08:00:00' (strings).
    Devuelve DataFrame con columnas: turno_id,start_time,end_time,break_min,lunch_min,duration_min,shift_type
    """
    cols = {c.lower(): c for c in df.columns}
    # id
    id_col = None
    for cand in ['turno_id','id','nombre','turno']:
        if cand in cols:
            id_col = cols[cand]; break
    # start/end
    start_col = None
    end_col = None
    for cand in ['start','hora_ingreso','hora_inicio','inicio','entrada','start_time']:
        if cand in cols:
            start_col = cols[cand]; break
    for cand in ['end','hora_fin','fin','salida','end_time','hora_salida']:
        if cand in cols:
            end_col = cols[cand]; break
    # breaks
    break_col = None
    lunch_col = None
    for cand in ['break_minutes','break','pausa_min','pausa','break_minutos']:
        if cand in cols:
            break_col = cols[cand]; break
    for cand in ['almuerzo_minutes','almuerzo','lunch_minutes','lunch','refrigerio','refrigerio_min']:
        if cand in cols:
            lunch_col = cols[cand]; break

    if start_col is None or end_col is None:
        raise ValueError("No se encontraron columnas de inicio/fin en la maestra de turnos. Nombres esperados: start/hora_ingreso/hora_inicio / end/hora_fin.")

    df2 = pd.DataFrame()
    df2['turno_id'] = df[id_col] if id_col is not None else df.index.astype(str)
    df2['start'] = df[start_col].astype(str)
    df2['end'] = df[end_col].astype(str)
    df2['break_min'] = df[break_col].fillna(0).astype(float) if break_col else 0
    df2['lunch_min'] = df[lunch_col].fillna(0).astype(float) if lunch_col else 0

    # Normalize times to minutes from midnight
    def time_to_min(t):
        try:
            # accept HH:MM or HH:MM:SS or numeric
            if isinstance(t,(int,float)):
                return int(t)
            t = t.strip()
            if t == '' or pd.isna(t):
                return 0
            parts = t.split(':')
            h = int(parts[0]); m = int(parts[1]) if len(parts)>1 else 0
            return h*60 + m
        except:
            # fallback
            return 0

    df2['start_min'] = df2['start'].apply(time_to_min)
    df2['end_min'] = df2['end'].apply(time_to_min)

    # handle overnight shifts (end < start)
    def compute_duration(s,e):
        if e >= s:
            return e - s
        else:
            return (24*60 - s) + e

    df2['duration_min'] = df2.apply(lambda r: compute_duration(r['start_min'], r['end_min']), axis=1)
    df2['effective_min'] = df2['duration_min'] - df2['break_min'] - df2['lunch_min']
    # classify day/night: day if start between 05:00 and 20:59
    def classify(smin):
        if 5*60 <= smin <= 20*60 + 59:
            return 'day'
        else:
            return 'night'
    df2['shift_type'] = df2['start_min'].apply(classify)
    return df2[['turno_id','start','end','start_min','end_min','duration_min','break_min','lunch_min','effective_min','shift_type']]

def load_forecast(df):
    """
    Busca columnas de timestamp y calls: aceptadas ['ds','timestamp','fecha','date','interval'],
    y ['y','calls','llamadas','predicted'].
    Devuelve DataFrame con datetime index y columna 'calls'
    """
    cols = {c.lower(): c for c in df.columns}
    time_col = None
    calls_col = None
    for cand in ['ds','timestamp','fecha','date','interval','date_time','datetime']:
        if cand in cols:
            time_col = cols[cand]; break
    for cand in ['y','calls','llamadas','predicted','forecast']:
        if cand in cols:
            calls_col = cols[cand]; break
    if time_col is None or calls_col is None:
        raise ValueError("El archivo de forecast debe tener columna de fecha (ds/timestamp) y llamadas (y/calls).")
    df2 = df[[time_col,calls_col]].copy()
    df2.columns = ['ds','calls']
    # parse dates
    df2['ds'] = pd.to_datetime(df2['ds'])
    df2 = df2.sort_values('ds').reset_index(drop=True)
    return df2

# ----------------------
# Erlang-C implementation
# ----------------------
def erlang_c_probability(a, n):
    """
    Probabilidad de espera en cola usando Erlang C components.
    a = offered load (lambda * aht) (Erlangs)
    n = servers (agents)
    Returns Erlang C (probability waiting)
    """
    if n <= 0:
        return 1.0
    # compute factorials with logs for stability
    # P0 denom
    sum_terms = 0.0
    for k in range(n):
        sum_terms += (a**k) / math.factorial(k)
    last_term = (a**n) / (math.factorial(n) * (1 - a / n)) if a < n else (a**n) / math.factorial(n) * 1e6
    p0 = 1.0 / (sum_terms + last_term)
    # Erlang C
    numerator = (a**n) / (math.factorial(n)) * (n / (n - a))
    erlangC = numerator * p0
    return min(max(erlangC, 0.0), 1.0)

def service_level_from_erlangC(erlangC, n, a, asa, aht):
    """
    Approximate service level: probability that waiting time <= ASA.
    Using exponential wait tail for M/M/n with Erlang C: P(W <= t) = 1 - erlangC * exp(- (n - a) * t / aht )
    """
    if n <= a:
        return 0.0
    exponent = - (n - a) * (asa) / aht
    return 1.0 - erlangC * math.exp(exponent)

def required_agents_for_interval(calls, aht_sec, service_level_target, interval_sec, asa=20):
    """
    Iteratively find minimum n such that service level >= target.
    calls: expected calls in interval
    aht_sec: average handling time in seconds
    interval_sec: seconds per interval (e.g., 1800)
    asa: acceptable speed of answer (seconds)
    """
    if calls <= 0:
        return 0
    # arrival rate per second = calls / interval_sec
    lam = calls / interval_sec
    # offered load a = lambda * aht
    a = lam * aht_sec
    # minimal n starts from ceil(a)
    n = max(1, int(math.ceil(a)))
    maxn = max(200, n + 200)  # safety limit
    while n <= maxn:
        erc = erlang_c_probability(a, n)
        sl = service_level_from_erlangC(erc, n, a, asa, aht_sec)
        if sl >= service_level_target:
            return n
        n += 1
    # if not found, return n
    return n

# ----------------------
# Roster optimizer (CP-SAT)
# ----------------------
def optimize_roster(turns_df, forecast_df, agents_available, aht_sec, absenteeism_pct, weekly_hours, service_level_target, interval_seconds):
    """
    turns_df: turn master parsed
    forecast_df: with 'ds' datetime and 'calls'
    agents_available: int
    Returns: schedule_df (agent_id, day, turno_id), interval_summary_df
    """
    # Build week days from forecast (group by date)
    forecast_df = forecast_df.copy()
    forecast_df['date'] = forecast_df['ds'].dt.date
    unique_days = sorted(forecast_df['date'].unique())[:7]  # limit to first 7 days (week)
    # Create intervals list for those days
    intervals = forecast_df[forecast_df['date'].isin(unique_days)].reset_index(drop=True)
    # required agents per interval
    required_agents = []
    for idx, row in intervals.iterrows():
        calls = float(row['calls'])
        req = required_agents_for_interval(calls=calls, aht_sec=aht_sec, service_level_target=service_level_target, interval_sec=interval_seconds)
        # adjust for absenteeism: required/(1-absent)
        if (1 - absenteeism_pct) > 0:
            req = math.ceil(req / (1 - absenteeism_pct))
        else:
            req = req
        required_agents.append(int(req))
    intervals['required_agents'] = required_agents

    # Precompute which shift covers which interval index for each day
    # We'll model candidate assignments: for each agent, day index, shift template -> covers subset of interval indices
    # Build list of day starts
    day_to_indices = {}
    for d in unique_days:
        day_to_indices[d] = list(intervals[intervals['date'] == d].index)

    # For simplicity, map shift times into minutes from midnight; then an interval timestamp maps to minute-of-day
    def minute_of_day(ts):
        return ts.hour*60 + ts.minute

    # coverage_map[(day, shift_idx)] = list of interval indices covered
    coverage_map = {}
    for d in unique_days:
        idxs = day_to_indices[d]
        for s_i, srow in turns_df.iterrows():
            covered = []
            for idx in idxs:
                ts = intervals.loc[idx,'ds']
                mod = minute_of_day(ts)
                s = int(srow['start_min']); e = int(srow['end_min'])
                if s <= e:
                    in_shift = (mod >= s and mod < e)
                else:
                    # overnight
                    in_shift = (mod >= s or mod < e)
                if in_shift:
                    covered.append(idx)
            coverage_map[(d, s_i)] = covered

    # Build CP model
    model = cp_model.CpModel()
    num_agents = agents_available
    days = list(range(len(unique_days)))
    shifts_idx = list(range(len(turns_df)))

    # Decision vars: x[a,d,s] = 1 if agent a assigned shift s on day d
    x = {}
    for a in range(num_agents):
        for d in days:
            for s in shifts_idx:
                x[(a,d,s)] = model.NewBoolVar(f"x_a{a}_d{d}_s{s}")

    # Each agent at most 1 shift per day
    for a in range(num_agents):
        for d in days:
            model.Add(sum(x[(a,d,s)] for s in shifts_idx) <= 1)

    # Weekly hours constraint per agent
    for a in range(num_agents):
        # total assigned minutes <= weekly_hours*60
        model.Add(sum(
            int(turns_df.loc[s,'duration_min']) * x[(a,d,s)]
            for d in days for s in shifts_idx
        ) <= int(weekly_hours * 60))

    # Day/Night consistency: define day_is_day[a,d], day_is_night[a,d]
    day_is_day = {}
    day_is_night = {}
    bigM = len(shifts_idx)
    for a in range(num_agents):
        for d in days:
            day_is_day[(a,d)] = model.NewBoolVar(f"day_is_day_{a}_{d}")
            day_is_night[(a,d)] = model.NewBoolVar(f"day_is_night_{a}_{d}")
            # sum of assignments to day shifts >= day_is_day
            model.Add(sum(x[(a,d,s)] for s in shifts_idx if turns_df.loc[s,'shift_type']=='day') >= day_is_day[(a,d)])
            model.Add(sum(x[(a,d,s)] for s in shifts_idx if turns_df.loc[s,'shift_type']=='day') <= bigM * day_is_day[(a,d)])
            model.Add(sum(x[(a,d,s)] for s in shifts_idx if turns_df.loc[s,'shift_type']=='night') >= day_is_night[(a,d)])
            model.Add(sum(x[(a,d,s)] for s in shifts_idx if turns_df.loc[s,'shift_type']=='night') <= bigM * day_is_night[(a,d)])
    # forbid day->night and night->day consecutive days
    for a in range(num_agents):
        for d in range(len(days)-1):
            model.Add(day_is_day[(a,d)] + day_is_night[(a,d+1)] <= 1)
            model.Add(day_is_night[(a,d)] + day_is_day[(a,d+1)] <= 1)

    # Coverage per interval: compute coverage_j = sum over a,d,s that cover interval j of x[a,d,s]
    coverage = {}
    for j in intervals.index:
        coverage[j] = model.NewIntVar(0, num_agents*10, f"coverage_{j}")
        # sum contributions
        terms = []
        for a in range(num_agents):
            for d_idx, d in enumerate(unique_days):
                for s in shifts_idx:
                    if j in coverage_map[(d, s)]:
                        terms.append(x[(a,d_idx,s)])
        if terms:
            model.Add(coverage[j] == sum(terms))
        else:
            model.Add(coverage[j] == 0)

    # shortfall per interval: shortfall_j >= required - coverage, >=0
    shortfall = {}
    for j in intervals.index:
        req = int(intervals.loc[j,'required_agents'])
        shortfall[j] = model.NewIntVar(0, max(req, num_agents)*10, f"shortfall_{j}")
        model.Add(shortfall[j] >= req - coverage[j])
        model.Add(shortfall[j] >= 0)

    # objective: minimize total shortfall, then minimize total assigned shifts
    total_shortfall = model.NewIntVar(0, 10**7, "total_shortfall")
    model.Add(total_shortfall == sum(shortfall[j] for j in intervals.index))
    total_assigned = model.NewIntVar(0, num_agents*len(days)*len(shifts_idx), "total_assigned")
    model.Add(total_assigned == sum(x.values()))

    # Weighted objective: minimize (W1 * shortfall + total_assigned)
    W1 = max(1, int(intervals['required_agents'].max())) * len(intervals) * 1000
    model.Minimize(total_shortfall * W1 + total_assigned)

    # Solve
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 30.0
    solver.parameters.num_search_workers = 8
    res = solver.Solve(model)

    assigned_rows = []
    if res == cp_model.OPTIMAL or res == cp_model.FEASIBLE:
        for a in range(num_agents):
            for d_idx, d in enumerate(unique_days):
                for s in shifts_idx:
                    if solver.Value(x[(a,d_idx,s)]) == 1:
                        # compute actual start/end datetimes
                        # find a sample datetime from intervals for day d
                        sample_ts = intervals[intervals['date']==d]['ds'].iloc[0]
                        start_min = int(turns_df.loc[s,'start_min'])
                        # build datetime
                        start_dt = datetime.combine(sample_ts.date(), datetime.min.time()) + timedelta(minutes=start_min)
                        # note end minute handling overnight
                        end_min = int(turns_df.loc[s,'end_min'])
                        if turns_df.loc[s,'end_min'] >= turns_df.loc[s,'start_min']:
                            end_dt = datetime.combine(sample_ts.date(), datetime.min.time()) + timedelta(minutes=end_min)
                        else:
                            # overnight: end next day
                            end_dt = datetime.combine(sample_ts.date(), datetime.min.time()) + timedelta(days=1, minutes=end_min)
                        assigned_rows.append({
                            'agent_id': f"ID_{a+1}",
                            'date': d,
                            'turno_id': turns_df.loc[s,'turno_id'],
                            'start': start_dt,
                            'end': end_dt,
                            'duration_min': turns_df.loc[s,'duration_min'],
                            'effective_min': turns_df.loc[s,'effective_min'],
                            'shift_type': turns_df.loc[s,'shift_type']
                        })
    schedule_df = pd.DataFrame(assigned_rows)

    # Build interval summary
    interval_rows = []
    for j in intervals.index:
        covered = solver.Value(coverage[j]) if (res == cp_model.OPTIMAL or res == cp_model.FEASIBLE) else 0
        req = int(intervals.loc[j,'required_agents'])
        calls = float(intervals.loc[j,'calls'])
        # Effective agents capacity per interval: each assigned agent provides interval_seconds * (effective_min/duration_min) seconds of work? Approximate:
        # We'll compute average break ratio across shifts covering interval and reduce capacity accordingly.
        # Simple approach: average effective_min / duration_min across shifts that could cover interval (not exact)
        # To avoid division by zero:
        avg_effective_ratio = 1.0
        # find shifts that cover j
        relevant_shifts = []
        for d_idx, d in enumerate(unique_days):
            for s in shifts_idx:
                if j in coverage_map[(d, s)]:
                    relevant_shifts.append(s)
        if relevant_shifts:
            ratios = [max(0.0, min(1.0, turns_df.loc[s,'effective_min'] / max(1.0, turns_df.loc[s,'duration_min']))) for s in relevant_shifts]
            avg_effective_ratio = np.mean(ratios) if len(ratios)>0 else 1.0
        # handled calls estimate:
        handled_calls = covered * (interval_seconds * avg_effective_ratio) / aht_sec
        short = max(0, req - covered)
        surplus = max(0, covered - req)
        interval_rows.append({
            'ds': intervals.loc[j,'ds'],
            'date': intervals.loc[j,'date'],
            'calls': calls,
            'required_agents': req,
            'covered_agents': covered,
            'handled_calls_est': handled_calls,
            'shortfall_agents': short,
            'surplus_agents': surplus
        })
    interval_summary = pd.DataFrame(interval_rows)
    return schedule_df, interval_summary, solver.StatusName(res)

# ----------------------
# Streamlit UI
# ----------------------
st.title("Mini-WFM: From Forecast → Staffing → Optimized Roster")
st.markdown("Carga tu maestra de turnos y el forecast (llamadas por intervalo). Ajusta TMO, ausentismo y agentes y corre el optimizador.")

col1, col2 = st.columns([1,1])
with col1:
    uploaded_turns = st.file_uploader("Maestra de turnos (Excel)", type=["xlsx","xls","csv"], key="turns")
with col2:
    uploaded_forecast = st.file_uploader("Forecast (llamadas por intervalo) (Excel)", type=["xlsx","xls","csv"], key="forecast")

# fallback to /mnt/data if no upload (useful si ya subiste)
use_default = False
default_turns_path = "/mnt/data/Mtr_Turnos.xlsx"
default_forecast_path = "/mnt/data/Input.xlsx"
if uploaded_turns is None and uploaded_forecast is None:
    if st.button("Cargar archivos por defecto (si existen en /mnt/data)"):
        use_default = True

st.sidebar.header("Parámetros operativos")
agents_available = st.sidebar.number_input("Agentes disponibles (N)", min_value=1, value=20, step=1)
aht_sec = st.sidebar.number_input("TMO / AHT (segundos)", min_value=10, value=300, step=1)
abs_percent = st.sidebar.slider("Ausentismo (%)", min_value=0.0, max_value=0.7, value=0.15, step=0.01)
weekly_hours = st.sidebar.number_input("Horas semanales por agente", min_value=1, value=44, step=1)
service_level = st.sidebar.slider("Nivel de servicio objetivo (ej: 0.8 = 80%)", min_value=0.5, max_value=0.99, value=0.8, step=0.01)
interval_seconds_input = st.sidebar.number_input("Segundos por intervalo (ej. 1800 para 30min)", min_value=60, value=1800, step=60)
asa = st.sidebar.number_input("ASA para SL (segundos)", min_value=1, value=20, step=1)

# Load data
turns_df = None
forecast_df = None
try:
    if uploaded_turns:
        if str(uploaded_turns.name).lower().endswith('.csv'):
            tmp = pd.read_csv(uploaded_turns)
        else:
            tmp = pd.read_excel(uploaded_turns)
        turns_df = load_master_turns(tmp)
    elif use_default:
        try:
            tmp = pd.read_excel(default_turns_path)
            turns_df = load_master_turns(tmp)
            st.success(f"Maestra de turnos cargada desde {default_turns_path}")
        except Exception as e:
            st.warning("No se pudo cargar archivo por defecto de maestra de turnos: " + str(e))
    if uploaded_forecast:
        if str(uploaded_forecast.name).lower().endswith('.csv'):
            tmp2 = pd.read_csv(uploaded_forecast)
        else:
            tmp2 = pd.read_excel(uploaded_forecast)
        forecast_df = load_forecast(tmp2)
    elif use_default:
        try:
            tmp2 = pd.read_excel(default_forecast_path)
            forecast_df = load_forecast(tmp2)
            st.success(f"Forecast cargado desde {default_forecast_path}")
        except Exception as e:
            st.warning("No se pudo cargar archivo por defecto de forecast: " + str(e))
except Exception as e:
    st.error("Error al leer archivos: " + str(e))

if turns_df is not None:
    with st.expander("Maestra de turnos (normalizada)"):
        st.dataframe(turns_df)

if forecast_df is not None:
    with st.expander("Forecast cargado (primeras filas)"):
        st.dataframe(forecast_df.head())

if st.button("Ejecutar optimización"):

    if turns_df is None or forecast_df is None:
        st.error("Necesito ambos archivos: maestra de turnos y forecast.")
    else:
        st.info("Ejecutando optimizador (toma ~30 segundos)...")
        schedule_df, interval_summary, status = optimize_roster(
            turns_df=turns_df,
            forecast_df=forecast_df,
            agents_available=int(agents_available),
            aht_sec=float(aht_sec),
            absenteeism_pct=float(abs_percent),
            weekly_hours=float(weekly_hours),
            service_level_target=float(service_level),
            interval_seconds=int(interval_seconds_input)
        )
        st.success(f"Optimización terminada — estado solver: {status}")
        if not schedule_df.empty:
            st.subheader("Asignación de turnos (schedule)")
            st.dataframe(schedule_df.sort_values(['agent_id','date']))

            csv = schedule_df.to_csv(index=False).encode('utf-8')
            st.download_button("Descargar asignación (CSV)", data=csv, file_name="schedule.csv", mime="text/csv")

        st.subheader("Resumen por intervalo (primeras filas)")
        st.dataframe(interval_summary.head(50))

        # Aggregate graphs
        fig, ax = plt.subplots(2,1, figsize=(10,6), sharex=True)
        # Agents required vs covered (by interval)
        ax[0].plot(interval_summary['ds'], interval_summary['required_agents'], label='Requeridos')
        ax[0].plot(interval_summary['ds'], interval_summary['covered_agents'], label='Conectados (optimizado)')
        ax[0].set_ylabel("Agentes")
        ax[0].legend()
        ax[0].grid(True)
        # Calls forecast vs handled
        ax[1].plot(interval_summary['ds'], interval_summary['calls'], label='Llamadas pronosticadas')
        ax[1].plot(interval_summary['ds'], interval_summary['handled_calls_est'], label='Estimadas atendidas')
        ax[1].set_ylabel("Llamadas")
        ax[1].legend()
        ax[1].grid(True)
        st.pyplot(fig)

        # Summary metrics
        total_short = interval_summary['shortfall_agents'].sum()
        total_surplus = interval_summary['surplus_agents'].sum()
        st.metric("Total faltantes (agentes sobre todos intervalos)", int(total_short))
        st.metric("Total sobrantes (agentes)", int(total_surplus))
        # Check capacity vs calls
        total_calls = interval_summary['calls'].sum()
        total_handled = interval_summary['handled_calls_est'].sum()
        st.metric("Llamadas pronosticadas (suma)", int(total_calls))
        st.metric("Llamadas estimadas a atender", int(total_handled))

        st.info("Notas: (1) Las pausas/almuerzos se consideran como reducción proporcional de capacidad (approx). (2) Si quieres programar tiempos exactos de break/almuerzo por intervalo hay que añadir lógica de ventanas y rotación de pausas, que podemos agregar en la siguiente iteración.")

2026-02-27 15:18:18.263 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-27 15:18:18.271 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-27 15:18:18.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-27 15:18:18.273 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-27 15:18:18.280 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-27 15:18:18.282 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-27 15:18:18.284 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-27 15:18:18.285 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar